## Setup
Credenciales y URIs

In [1]:
import sys

!{sys.executable} -m pip install "sagemaker>=2.99.0,<3.0"

import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

sagemaker_session = sagemaker.session.Session()
region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
default_bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
default_bucket_prefix_path = ""

if default_bucket_prefix:
    default_bucket_prefix_path = f"/{default_bucket_prefix}"

model_package_group_name = f"AbaloneModelPackageGroupDev"

# Imágenes en ECR
account_id = boto3.client("sts").get_caller_identity()["Account"]
repo_processing = "sagemaker-sklearn-preprocess"
repo_training = "sagemaker-xgboost-byoc"

# URIs dinámicas
processing_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_processing}:latest"
training_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_training}:latest"

print(f"Bucket por defecto: {default_bucket}")
print(f"Processing Image: {processing_image_uri}")
print(f"Training Image: {training_image_uri}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket por defecto: sagemaker-us-east-1-150215480648
Processing Image: 150215480648.dkr.ecr.us-east-1.amazonaws.com/sagemaker-sklearn-preprocess:latest
Training Image: 150215480648.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost-byoc:latest


## Carga de datos y parámetros

In [2]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)
# Bucket con los datos cargados en la tarea 6
bucket = default_bucket
input_prefix = "sagemaker/processing-data/input/raw"
s3_data_uri = f"s3://{bucket}/{input_prefix}"

print(f"El pipeline usará los datos contenidos en : {s3_data_uri}")

# Verificación de archivos en bucket
response = sagemaker_session.boto_session.client("s3").list_objects_v2(
    Bucket=bucket, 
    Prefix=input_prefix
)
print("\nArchivos detectados en S3:")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f" {obj['Key']}")

# Parámetros
processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")


input_data = ParameterString(name="InputData", default_value=s3_data_uri)
batch_data = ParameterString(name="BatchData", default_value=s3_data_uri)
rmse_threshold = ParameterFloat(name="RmseThreshold", default_value=5.0)

El pipeline usará los datos contenidos en : s3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw

Archivos detectados en S3:
 sagemaker/processing-data/input/raw/item_categories.csv
 sagemaker/processing-data/input/raw/item_categories_en.csv
 sagemaker/processing-data/input/raw/items.csv
 sagemaker/processing-data/input/raw/items_en.csv
 sagemaker/processing-data/input/raw/sales_train.csv
 sagemaker/processing-data/input/raw/shops.csv
 sagemaker/processing-data/input/raw/shops_en.csv
 sagemaker/processing-data/input/raw/submission.csv
 sagemaker/processing-data/input/raw/test.csv


## PrepocessingStep

In [7]:
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep


script_processor = ScriptProcessor(
    image_uri=processing_image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name="byoc-preprocess",
    role=role,
    sagemaker_session=pipeline_session,
)


processor_args = script_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data, 
            destination="/opt/ml/processing/input"
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/output/test"),
    ],
    code="../processing/container/preprocess.py",
)


step_process = ProcessingStep(name="PreprocesamientoBYOC", step_args=processor_args)
print("Preprocessing step listo")

Preprocessing step listo


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## TrainingStep

In [8]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

model_path = f"s3://{default_bucket}/{default_bucket_prefix}/modelos"

xgb_train = Estimator(
    image_uri=training_image_uri,
    instance_type=instance_type,
    instance_count=1,
    output_path=model_path,
    role=role,
    sagemaker_session=pipeline_session,
)


train_args = xgb_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
    }
)


step_train = TrainingStep(
    name="EntrenamientoBYOC",
    step_args=train_args,
)
print("TrainingStep listo")

TrainingStep listo


## EvaluationStep

In [10]:
from sagemaker.workflow.properties import PropertyFile

eval_args = script_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="../processing/container/evaluate.py",
)


evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)


step_eval = ProcessingStep(
    name="EvaluacionBYOC",
    step_args=eval_args,
    property_files=[evaluation_report],
)
print("EvaluationStep listo")

EvaluationStep listo
